In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import Annotated
from typing_extensions import TypedDict, Literal
from langgraph.graph.message import add_messages

load_dotenv()
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
print("✅ Prêt")

✅ Prêt


In [2]:
class MedicalState(TypedDict, total=False):
    messages: Annotated[list, add_messages]
    next: Literal["diagnostic_agent", "physician_review", "report_agent", "FINISH"]
    patient_case: str
    question_count: int
    patient_answers: list
    diagnostic_summary: str
    interim_care: str
    physician_treatment: str
    final_report: str

def supervisor(state: MedicalState) -> MedicalState:
    next_node = state.get("next", "diagnostic_agent")
    print(f"🔀 Supervisor → {next_node}")
    return {"next": next_node}

def diagnostic_agent(state: MedicalState) -> MedicalState:
    question_count = state.get("question_count", 0)
    patient_answers = state.get("patient_answers", [])
    patient_case = state.get("patient_case", "")

    if question_count < 5:
        prompt = f"""Tu es un agent médical. Pose UNE SEULE question courte au patient.
Cas : {patient_case}
Questions précédentes : {patient_answers}
Question {question_count + 1}/5 — différente des précédentes."""
        question = llm.invoke([HumanMessage(content=prompt)]).content
        print(f"❓ Question {question_count + 1}: {question}")
        patient_response = input("👤 Réponse : ")
        new_answers = patient_answers + [f"Q{question_count+1}: {question} → {patient_response}"]
        return {"question_count": question_count + 1, "patient_answers": new_answers, "next": "diagnostic_agent"}
    else:
        print("🔬 Génération synthèse...")
        synthese = llm.invoke([HumanMessage(content=f"""Synthèse clinique préliminaire PRUDENTE pour :
Cas : {patient_case}
Réponses : {patient_answers}
3-4 phrases maximum.""")]).content
        interim = llm.invoke([HumanMessage(content=f"Recommandation intermédiaire générale basée sur : {synthese}. 2 phrases max.")]).content
        print("✅ Synthèse prête")
        return {"diagnostic_summary": synthese, "interim_care": interim, "next": "physician_review"}

def physician_review(state: MedicalState) -> MedicalState:
    # Cet agent est vide — l'interruption se passe AVANT lui
    print("👨‍⚕️ Médecin en train de valider...")
    return state

def report_agent(state: MedicalState) -> MedicalState:
    answers_text = "\n".join(state.get("patient_answers", []))
    rapport = f"""
╔══════════════════════════════════════╗
       RAPPORT CLINIQUE PRÉLIMINAIRE        
╚══════════════════════════════════════╝

📌 CAS : {state.get('patient_case')}

📝 RÉPONSES PATIENT :
{answers_text}

🔬 SYNTHÈSE CLINIQUE :
{state.get('diagnostic_summary')}

💊 RECOMMANDATION INTERMÉDIAIRE :
{state.get('interim_care')}

👨‍⚕️ TRAITEMENT MÉDECIN :
{state.get('physician_treatment')}

⚠️  Ce système ne remplace pas une consultation médicale.
"""
    print(rapport)
    return {"final_report": rapport, "next": "FINISH"}

print("✅ Agents définis")

✅ Agents définis


In [3]:
# MemorySaver = mémoire pour pouvoir reprendre après l'interruption
memory = MemorySaver()

builder = StateGraph(MedicalState)
builder.add_node("supervisor", supervisor)
builder.add_node("diagnostic_agent", diagnostic_agent)
builder.add_node("physician_review", physician_review)
builder.add_node("report_agent", report_agent)
builder.set_entry_point("supervisor")

builder.add_conditional_edges(
    "supervisor",
    lambda s: s.get("next"),
    {
        "diagnostic_agent": "diagnostic_agent",
        "physician_review": "physician_review",
        "report_agent": "report_agent",
        "FINISH": END
    }
)
builder.add_edge("diagnostic_agent", "supervisor")
builder.add_edge("physician_review", "supervisor")
builder.add_edge("report_agent", "supervisor")

# ⚡ INTERRUPTION avant physician_review
graph = builder.compile(
    checkpointer=memory,
    interrupt_before=["physician_review"]
)

print("✅ Graphe compilé avec interruption HITL")

✅ Graphe compilé avec interruption HITL


In [4]:
import uuid
thread_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": thread_id}}

print("=== PHASE 1 : CONSULTATION PATIENT ===\n")
graph.invoke({
    "patient_case": "Patient de 45 ans, douleurs thoraciques depuis ce matin.",
    "question_count": 0,
    "patient_answers": [],
    "next": "diagnostic_agent"
}, config=config)

print("\n⏸️  WORKFLOW EN PAUSE — en attente du médecin")

=== PHASE 1 : CONSULTATION PATIENT ===

🔀 Supervisor → diagnostic_agent
❓ Question 1: Avez-vous ressenti d'autres symptômes tels que la fièvre, la toux ou une difficulté à respirer ?
🔀 Supervisor → diagnostic_agent
❓ Question 2: Avez-vous eu des accidents ou des chutes récents qui pourraient avoir contribué à vos douleurs thoraciques ?
🔀 Supervisor → diagnostic_agent
❓ Question 3: Q3 : Avez-vous remarqué une différence dans la nature de vos douleurs thoraciques depuis ce matin, par exemple si elles sont plus intenses ou si elles se déplacent vers une autre partie du thorax ?
🔀 Supervisor → diagnostic_agent
❓ Question 4: Q4 : Avez-vous des antécédents de problèmes cardiaques ou vasculaires, tels que des maladies coronaires, des accidents vasculaires cérébraux ou des problèmes de circulation sanguine ?
🔀 Supervisor → diagnostic_agent
❓ Question 5: Q5 : Avez-vous des douleurs thoraciques qui se déclenchent ou s'intensifient lors de mouvements spécifiques, tels que la respiration profonde 

In [5]:
print("=== PHASE 2 : INTERVENTION MÉDECIN ===\n")

# Voir la synthèse
current_state = graph.get_state(config)
print("📋 Synthèse :", current_state.values.get("diagnostic_summary"))
print("💊 Recommandation :", current_state.values.get("interim_care"))

# Le médecin saisit son traitement
treatment = input("\n👨‍⚕️ Médecin — traitement proposé : ")

# Mettre à jour le state avec le traitement
graph.update_state(config, {"physician_treatment": treatment, "next": "report_agent"})

# Reprendre le workflow
print("\n=== PHASE 3 : GÉNÉRATION DU RAPPORT ===\n")
graph.invoke(None, config=config)

=== PHASE 2 : INTERVENTION MÉDECIN ===

📋 Synthèse : **Synthèse clinique préliminaire PRUDENTE**

**Patient de 45 ans, douleurs thoraciques depuis ce matin**

- **Q1 :** Le patient n'a pas ressenti de fièvre, de toux ou de difficulté à respirer.
- **Q2 :** Le patient n'a pas eu d'accidents ou de chutes récents.
- **Q3 :** Le patient a remarqué que ses douleurs thoraciques sont restées constantes et ne se sont pas déplacées.
- **Q4 :** Le patient n'a pas d'antécédents de problèmes cardiaques ou vasculaires.
- **Q5 :** Le patient n'a pas de douleurs thoraciques qui se déclenchent ou s'intensifient lors de mouvements spécifiques.

**Diagnostic préliminaire :** Douleurs thoraciques non spécifiques, nécessitant une évaluation plus approfondie pour exclure les causes sous-jacentes.
💊 Recommandation : **Recommandation intermédiaire générale :**

- Effectuer un examen physique complet pour évaluer la douleur thoracique et rechercher d'éventuelles anomalies.
- Réaliser des examens de laboratoir

{'messages': [],
 'next': 'FINISH',
 'patient_case': 'Patient de 45 ans, douleurs thoraciques depuis ce matin.',
 'question_count': 5,
 'patient_answers': ["Q1: Avez-vous ressenti d'autres symptômes tels que la fièvre, la toux ou une difficulté à respirer ? → ",
  'Q2: Avez-vous eu des accidents ou des chutes récents qui pourraient avoir contribué à vos douleurs thoraciques ? → ',
  'Q3: Q3 : Avez-vous remarqué une différence dans la nature de vos douleurs thoraciques depuis ce matin, par exemple si elles sont plus intenses ou si elles se déplacent vers une autre partie du thorax ? → ',
  'Q4: Q4 : Avez-vous des antécédents de problèmes cardiaques ou vasculaires, tels que des maladies coronaires, des accidents vasculaires cérébraux ou des problèmes de circulation sanguine ? → ',
  "Q5: Q5 : Avez-vous des douleurs thoraciques qui se déclenchent ou s'intensifient lors de mouvements spécifiques, tels que la respiration profonde ou la marche ? → "],
 'diagnostic_summary': "**Synthèse clini